In [8]:
import pandas as pd

regression_df = pd.read_csv(
    r"C:\Users\palla\OneDrive\Documents\Coding Projects\Hospital Diabetes Dataset\hospital-readmissions-sql-eda\outputs\06_model_dataset_full.csv"
)

print(regression_df.head())
print(regression_df.info())
print(regression_df.describe())


   readmit_binary      age  gender             race  time_in_hospital  \
0               0   [0-10)  Female        Caucasian                 1   
1               0  [10-20)  Female        Caucasian                 3   
2               0  [20-30)  Female  AfricanAmerican                 2   
3               0  [30-40)    Male        Caucasian                 2   
4               0  [40-50)    Male        Caucasian                 1   

   num_lab_procedures  num_procedures  num_medications  number_outpatient  \
0                  41               0                1                  0   
1                  59               0               18                  0   
2                  11               5               13                  2   
3                  44               1               16                  0   
4                  51               0                8                  0   

   number_emergency  number_inpatient  number_diagnoses A1Cresult  \
0                 0          

In [9]:
print(regression_df.shape)

print("\nTarget distribution:")
print(regression_df["readmit_binary"].value_counts())
print("\nTarget proportion:")
print(regression_df["readmit_binary"].value_counts(normalize=True))


(101766, 20)

Target distribution:
readmit_binary
0    90409
1    11357
Name: count, dtype: int64

Target proportion:
readmit_binary
0    0.888401
1    0.111599
Name: proportion, dtype: float64


In [10]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    average_precision_score
)

# -----------------------------
# 1) Define features + target
# -----------------------------
X_features = regression_df.drop(columns=["readmit_binary"])
y_target = regression_df["readmit_binary"].astype(int)

categorical_columns = X_features.columns.tolist()

# -----------------------------
# 2) Preprocess + model pipeline
# -----------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_columns)
    ],
    remainder="drop"
)

logistic_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

model_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", logistic_model)
])

# -----------------------------
# 3) Train/test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_features,
    y_target,
    test_size=0.2,
    random_state=42,
    stratify=y_target
)

# -----------------------------
# 4) Fit
# -----------------------------
model_pipeline.fit(X_train, y_train)

# -----------------------------
# 5) Predict probabilities
# -----------------------------
probabilities = model_pipeline.predict_proba(X_test)[:, 1]

# -----------------------------
# 6) Core metrics (threshold-free)
# -----------------------------
roc_auc = roc_auc_score(y_test, probabilities)
pr_auc = average_precision_score(y_test, probabilities)

print("ROC AUC:", round(roc_auc, 4))
print("PR AUC (Average Precision):", round(pr_auc, 4))

# -----------------------------
# 7) Threshold option A: 0.50
# -----------------------------
threshold_50 = 0.50
pred_50 = (probabilities >= threshold_50).astype(int)

print("\n--- Threshold = 0.50 ---")
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_50))
print("\nClassification Report:\n", classification_report(y_test, pred_50))

# -----------------------------
# 8) Threshold option B: top 10% risk (capacity-based)
# -----------------------------
top_pct = 0.10
cutoff_index = int(len(probabilities) * (1 - top_pct))  # 90th percentile cutoff
threshold_top10 = np.sort(probabilities)[cutoff_index]

pred_top10 = (probabilities >= threshold_top10).astype(int)

print("\n--- Threshold = Top 10% Risk ---")
print("Top 10% threshold value:", round(float(threshold_top10), 4))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_top10))
print("\nClassification Report:\n", classification_report(y_test, pred_top10))

# Extra: what is the actual readmission rate among top 10%?
risk_df = pd.DataFrame({"actual": y_test.values, "proba": probabilities})
top10 = risk_df.sort_values("proba", ascending=False).head(int(len(risk_df) * top_pct))
print("\nReadmission rate in top 10% highest-risk group:", round(top10["actual"].mean(), 4))
print("Overall readmission rate (test set):", round(y_test.mean(), 4))


ROC AUC: 0.6722
PR AUC (Average Precision): 0.2155

--- Threshold = 0.50 ---
Confusion Matrix:
 [[12249  5834]
 [  970  1301]]

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.68      0.78     18083
           1       0.18      0.57      0.28      2271

    accuracy                           0.67     20354
   macro avg       0.55      0.63      0.53     20354
weighted avg       0.84      0.67      0.73     20354


--- Threshold = Top 10% Risk ---
Top 10% threshold value: 0.6613
Confusion Matrix:
 [[16588  1495]
 [ 1730   541]]

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.92      0.91     18083
           1       0.27      0.24      0.25      2271

    accuracy                           0.84     20354
   macro avg       0.59      0.58      0.58     20354
weighted avg       0.83      0.84      0.84     20354


Readmission rate in top 10% highest-risk group: 0.26

c:\Users\palla\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [4, 6, 7, 9, 18] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# 1) Load
full_regression_df = pd.read_csv(
    r"C:\Users\palla\OneDrive\Documents\Coding Projects\Hospital Diabetes Dataset\hospital-readmissions-sql-eda\outputs\06_model_dataset_full.csv"
)

# 2) Basic cleanup: treat blank strings as missing
full_regression_df = full_regression_df.replace(r"^\s*$", np.nan, regex=True)

# 3) Split X/y
X_features = full_regression_df.drop(columns=["readmit_binary"])
y_target = full_regression_df["readmit_binary"].astype(int)

# 4) Define numeric vs categorical columns
numeric_cols = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

categorical_cols = [c for c in X_features.columns if c not in numeric_cols]

# 5) Preprocess + model
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("scaler", StandardScaler())
        ]), numeric_cols),
        ("cat", Pipeline(steps=[
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols),
    ]
)

logistic_model = LogisticRegression(
    max_iter=3000,
    class_weight="balanced"
)

model_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", logistic_model)
])

# 6) Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_features,
    y_target,
    test_size=0.2,
    random_state=42,
    stratify=y_target
)

# 7) Fit
model_pipeline.fit(X_train, y_train)

# 8) Probabilities
probabilities = model_pipeline.predict_proba(X_test)[:, 1]

# 9) Threshold-free metrics
roc_auc = roc_auc_score(y_test, probabilities)
pr_auc = average_precision_score(y_test, probabilities)

print("ROC AUC:", round(roc_auc, 4))
print("PR AUC (Average Precision):", round(pr_auc, 4))

# ----------------------------
# Threshold A: 0.50 (reference)
# ----------------------------
threshold_50 = 0.50
pred_50 = (probabilities >= threshold_50).astype(int)

print("\n--- Threshold = 0.50 ---")
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_50))
print("\nClassification Report:\n", classification_report(y_test, pred_50))

# --------------------------------------
# Threshold B: Top 10% risk (operational)
# --------------------------------------
top_pct = 0.10
cutoff_index = int(len(probabilities) * (1 - top_pct))  # 90th percentile
threshold_top10 = np.sort(probabilities)[cutoff_index]
pred_top10 = (probabilities >= threshold_top10).astype(int)

print("\n--- Threshold = Top 10% Risk ---")
print("Top 10% threshold value:", round(float(threshold_top10), 4))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_top10))
print("\nClassification Report:\n", classification_report(y_test, pred_top10))

risk_df = pd.DataFrame({"actual": y_test.values, "proba": probabilities})
top10 = risk_df.sort_values("proba", ascending=False).head(int(len(risk_df) * top_pct))

print("\nReadmission rate in top 10% highest-risk group:", round(top10["actual"].mean(), 4))
print("Overall readmission rate (test set):", round(y_test.mean(), 4))


ROC AUC: 0.6745
PR AUC (Average Precision): 0.2186

--- Threshold = 0.50 ---
Confusion Matrix:
 [[12399  5684]
 [  989  1282]]

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.69      0.79     18083
           1       0.18      0.56      0.28      2271

    accuracy                           0.67     20354
   macro avg       0.56      0.63      0.53     20354
weighted avg       0.84      0.67      0.73     20354


--- Threshold = Top 10% Risk ---
Top 10% threshold value: 0.6441
Confusion Matrix:
 [[16595  1488]
 [ 1723   548]]

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.92      0.91     18083
           1       0.27      0.24      0.25      2271

    accuracy                           0.84     20354
   macro avg       0.59      0.58      0.58     20354
weighted avg       0.83      0.84      0.84     20354


Readmission rate in top 10% highest-risk group: 0.26

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

import plotly.express as px
import plotly.graph_objects as go

# Use already-fit pipeline: model_pipeline
# And existing X_features, y_target, X_train, X_test, y_train, y_test, probabilities

# 1) Transform X_test into the model’s numeric feature space (after preprocessing)
X_test_transformed = model_pipeline.named_steps["preprocess"].transform(X_test)

# If this becomes a sparse matrix, convert to dense for PCA (ok for moderate sizes)
try:
    X_test_dense = X_test_transformed.toarray()
except AttributeError:
    X_test_dense = X_test_transformed

# 2) PCA to 2D
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_test_dense)

pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "actual": y_test.values,
    "proba": probabilities,
})

# Add original features for hover 

hover_cols = [
    "time_in_hospital",
    "num_medications",
    "number_inpatient",
    "age",            # if you have it
    "A1Cresult"       # if you have it
]
hover_cols = [c for c in hover_cols if c in X_test.columns]

for c in hover_cols:
    pca_df[c] = X_test[c].values

# 3) Interactive scatter
fig = px.scatter(
    pca_df,
    x="PC1",
    y="PC2",
    color="proba",              # continuous color = predicted risk
    symbol="actual",            # shape = actual label 0/1
    hover_data=["proba", "actual"] + hover_cols,
    title="Patients projected to 2D (PCA) with predicted readmission risk"
)

# 4) Add an approximate “decision boundary” line/contour on the PCA plane
#    Create a grid in PCA space, mapping back to original space approximately,
#    then scoring the model. (Approximate because inverse PCA loses information.)

# Build grid
pc1_min, pc1_max = pca_df["PC1"].quantile([0.01, 0.99]).values
pc2_min, pc2_max = pca_df["PC2"].quantile([0.01, 0.99]).values

grid_n = 140
pc1 = np.linspace(pc1_min, pc1_max, grid_n)
pc2 = np.linspace(pc2_min, pc2_max, grid_n)
PC1, PC2 = np.meshgrid(pc1, pc2)
grid_pca = np.c_[PC1.ravel(), PC2.ravel()]

# Inverse PCA back to the preprocessed feature space (approx)
grid_dense = pca.inverse_transform(grid_pca)

# Score the logistic model directly on preprocessed features
log_model = model_pipeline.named_steps["model"]
grid_proba = log_model.predict_proba(grid_dense)[:, 1].reshape(PC1.shape)

# Choose threshold (0.5 or top10 threshold you computed earlier)
threshold = 0.5  # or: threshold_top10

# Add contour line where proba == threshold
fig.add_trace(
    go.Contour(
        x=pc1,
        y=pc2,
        z=grid_proba,
        showscale=False,
        contours=dict(
            start=threshold, end=threshold, size=1e-6, coloring="none"
        ),
        line=dict(width=3),
        hoverinfo="skip",
        name=f"Boundary p={threshold}"
    )
)

fig.update_layout(
    height=800,
    width=1000
)

fig.update_traces(marker=dict(size=4, opacity=0.25), selector=dict(type="scatter"))
fig.update_traces(marker=dict(size=4, opacity=0.25), selector=dict(type="scattergl"))
fig.show()


In [27]:
feature_names = model_pipeline.named_steps["preprocess"].get_feature_names_out()

pca_table = pd.DataFrame(
    pca.components_,
    columns=feature_names
)
pca_table.iloc[0].abs().sort_values(ascending=False).head(10)

top_features = {}

for i in range(2):
    component = pca_table.iloc[i]
    
    top_features[f"PC{i+1}"] = component.abs().sort_values(ascending=False).head(10).index.tolist()

pd.DataFrame(dict([(k,pd.Series(v)) for k,v in top_features.items()]))


,PC1,PC2
0,num__num_medications,num__number_inpatient
1,num__time_in_hospital,num__num_procedures
2,num__num_lab_procedures,num__number_outpatient
3,num__number_diagnoses,num__number_diagnoses
4,num__num_procedures,num__number_emergency
5,num__number_inpatient,cat__admission_source_id_7
6,cat__discharge_disposition_id_1,cat__admission_type_id_1
7,cat__change_No,num__num_medications
8,cat__change_Ch,num__num_lab_procedures
9,cat__insulin_No,cat__admission_source_id_1


In [28]:
import numpy as np
import pandas as pd

rows = []
for i in range(2):
    comp = pca_table.iloc[i]
    top = comp.reindex(comp.abs().sort_values(ascending=False).head(10).index)

    for feat, val in top.items():
        rows.append({
            "PC": f"PC{i+1}",
            "feature": feat,
            "loading": float(val),
            "abs_loading": float(abs(val))
        })

tidy = pd.DataFrame(rows).sort_values(["PC", "abs_loading"], ascending=[True, False])
tidy

,PC,feature,loading,abs_loading
0,PC1,num__num_medications,0.565699,0.565699
1,PC1,num__time_in_hospital,0.505957,0.505957
2,PC1,num__num_lab_procedures,0.367009,0.367009
3,PC1,num__number_diagnoses,0.339790,0.339790
4,PC1,num__num_procedures,0.323296,0.323296
5,PC1,num__number_inpatient,0.119677,0.119677
6,PC1,cat__discharge_disposition_id_1,-0.092800,0.092800
7,PC1,cat__change_No,-0.092759,0.092759
8,PC1,cat__change_Ch,0.092759,0.092759
9,PC1,cat__insulin_No,-0.087288,0.087288
